# Performance Benchmark: DuckReg vs Pyfixest

Benchmarks **running time** across N × K × nFE × model type × package.

### Files
| File | Role |
|------|------|
| [`scripts/run_single.py`](scripts/run_single.py) | Runs one (package, model, N, K, nFE) combination; writes `results/<key>.json` |
| [`scripts/run_single_benchmark.sh`](scripts/run_single_benchmark.sh) | SLURM wrapper — invoked by orchestrator |
| [`scripts/orchestrate.py`](scripts/orchestrate.py) | Submits jobs (max 4 concurrent), tracks `results/manifest.csv`, waits for completion |
| [`scripts/collect_results.py`](scripts/collect_results.py) | Merges JSON results; flags OOM kills from `.err` logs |

**Cell 2** checks for `results/benchmark_results_large.csv`.  
- If found → loads and displays.  
- If not → runs `python scripts/orchestrate.py` (which submits jobs and waits) then collects.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

## 1. Check Results / Submit Benchmark Jobs

In [ ]:
import subprocess
from pathlib import Path

BENCH_DIR   = Path(".")
RESULTS_DIR = BENCH_DIR / "results"
RESULTS_CSV = RESULTS_DIR / "benchmark_results_large.csv"

results = None

if RESULTS_CSV.exists():
    results = pd.read_csv(RESULTS_CSV)
    n_ok  = (results["status"] == "success").sum()
    n_oom = (results["status"] == "oom_killed").sum()
    n_fail = (results["status"] == "failed").sum()
    print(f"Results loaded from {RESULTS_CSV}  ({len(results)} rows)")
    print(f"  success={n_ok}  oom_killed={n_oom}  failed={n_fail}")
    display(results)
else:
    print(f"No results at {RESULTS_CSV}.")
    print("Launching orchestrate.py — submitting SLURM jobs (max 4 concurrent) and waiting …")
    proc = subprocess.run(
        ["python", "scripts/orchestrate.py", "--results-dir", "results"],
        cwd=str(BENCH_DIR),
    )
    if proc.returncode == 0 and RESULTS_CSV.exists():
        results = pd.read_csv(RESULTS_CSV)
        print(f"\nDone.  {len(results)} rows collected.")
        display(results)
    else:
        print("orchestrate.py finished.  Re-run this cell to load results.")

No results at benchmark_results_large.csv.
Launching orchestrate.py — submitting SLURM jobs (max 4 concurrent) and waiting …
Total combos : 216
Already done : 4
To submit    : 212
  [1/212] submitted job 6330296: duckreg iv N=100000 K=5 nFE=100
  [2/212] submitted job 6330297: pyfixest iv N=100000 K=5 nFE=100
  [3/212] submitted job 6330298: duckreg pooled N=100000 K=5 nFE=1000
  [4/212] submitted job 6330299: pyfixest pooled N=100000 K=5 nFE=1000
  4 jobs running (max 4), waiting 30s …
  [5/212] submitted job 6330303: duckreg fe N=100000 K=5 nFE=1000
  [6/212] submitted job 6330304: pyfixest fe N=100000 K=5 nFE=1000
  [7/212] submitted job 6330305: duckreg iv N=100000 K=5 nFE=1000
  [8/212] submitted job 6330306: pyfixest iv N=100000 K=5 nFE=1000
  4 jobs running (max 4), waiting 30s …
  [9/212] submitted job 6330309: duckreg pooled N=100000 K=5 nFE=10000
  [10/212] submitted job 6330310: pyfixest pooled N=100000 K=5 nFE=10000
  [11/212] submitted job 6330311: duckreg fe N=100000 K=5 

,package,model_type,N,K,nFE1,nFE2,job_id,status,time_seconds,error
0,duckreg,fe,10000000,10,100,100,6334334,failed,NaN,Catalog Error: unrecognized configuration para...
1,duckreg,fe,10000000,10,1000,1000,6334341,failed,NaN,Catalog Error: unrecognized configuration para...
2,duckreg,fe,10000000,10,10000,10000,6334350,failed,NaN,Catalog Error: unrecognized configuration para...
3,duckreg,fe,10000000,20,100,100,6334360,failed,NaN,Catalog Error: unrecognized configuration para...
4,duckreg,fe,10000000,20,1000,1000,6336276,failed,NaN,Catalog Error: unrecognized configuration para...
...,...,...,...,...,...,...,...,...,...,...
211,pyfixest,pooled,50000000,20,10000,10000,6336359,oom_killed,NaN,OOM killed
212,duckreg,fe,50000000,20,10000,10000,6336361,oom_killed,NaN,OOM killed
213,pyfixest,fe,50000000,20,10000,10000,6336362,oom_killed,NaN,OOM killed
214,duckreg,iv,50000000,20,10000,10000,6336364,oom_killed,NaN,OOM killed


## 2. Visualize Results

In [ ]:
if results is None:
    print("No results — run cell 1 first.")
else:
    df_ok = results[results["status"] == "success"].copy()

    # OOM / failure summary
    df_bad = results[results["status"] != "success"]
    if not df_bad.empty:
        print("Non-success runs:")
        display(df_bad[["package", "model_type", "N", "K", "nFE1", "status", "error", "job_id"]])

    if df_ok.empty:
        print("No successful runs to plot yet.")
    else:
        def plot_time_comparison(df_ok):
            pivot = df_ok.pivot_table(
                values="time_seconds",
                index=["N", "K", "nFE1", "model_type"],
                columns="package",
                aggfunc="mean",
            ).reset_index()
            for pkg in ("duckreg", "pyfixest"):
                if pkg not in pivot.columns:
                    pivot[pkg] = float("nan")
            pivot["speedup"] = pivot["pyfixest"] / pivot["duckreg"]

            model_types = [m for m in ["pooled", "fe", "iv"]
                           if m in pivot["model_type"].unique()]
            fig, axes = plt.subplots(1, len(model_types),
                                     figsize=(7 * len(model_types), 5))
            if len(model_types) == 1:
                axes = [axes]

            for ax, mt in zip(axes, model_types):
                data = pivot[pivot["model_type"] == mt].sort_values(["N", "K"])
                if data.empty:
                    continue
                x_labels = [
                    f"N={r['N']//1000}k\nK={r['K']}\nnFE={r['nFE1']}"
                    for _, r in data.iterrows()
                ]
                x = np.arange(len(x_labels))
                w = 0.35
                ax.bar(x - w/2, data["duckreg"],  w, label="duckreg",  alpha=0.8)
                ax.bar(x + w/2, data["pyfixest"], w, label="pyfixest", alpha=0.8)
                ax.set_xlabel("Configuration")
                ax.set_ylabel("Time (seconds)")
                ax.set_title(f"{mt.upper()} Model")
                ax.set_xticks(x)
                ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
                ax.legend()
                ax.grid(axis="y", alpha=0.3)

            plt.tight_layout()
            plt.savefig("benchmark_time_comparison.png", dpi=150, bbox_inches="tight")
            plt.show()
            return pivot

        pivot = plot_time_comparison(df_ok)
        display(pivot[["N", "K", "nFE1", "model_type", "duckreg", "pyfixest", "speedup"]])

## 3. Summary Statistics

In [ ]:
if results is None:
    print("No results — run cell 1 first.")
else:
    df_ok = results[results["status"] == "success"].copy()

    if df_ok.empty:
        print("No successful runs yet.")
    else:
        # Time summary
        time_summary = df_ok.groupby(["package", "model_type"]).agg(
            time_mean=("time_seconds", "mean"),
            time_std =("time_seconds", "std"),
            time_min =("time_seconds", "min"),
            time_max =("time_seconds", "max"),
            n        =("time_seconds", "count"),
        ).round(2)
        print("Time summary by package and model type:")
        display(time_summary)

        # Speedup
        speedup = df_ok.pivot_table(
            values="time_seconds",
            index=["model_type", "N", "K", "nFE1"],
            columns="package",
        ).reset_index()
        if "duckreg" in speedup.columns and "pyfixest" in speedup.columns:
            speedup["speedup"] = speedup["pyfixest"] / speedup["duckreg"]
            print("\nAverage speedup (pyfixest / duckreg) by model type:")
            display(speedup.groupby("model_type")["speedup"].mean().round(2))
            print(f"\nOverall average speedup: {speedup['speedup'].mean():.2f}x")

    # Status summary across all runs
    print("\nAll runs by status:")
    display(results.groupby(["package", "status"]).size().rename("count").reset_index())


Average Speedup by Model Type:


model_type
fe    0.17
Name: speedup, dtype: float64


Overall average speedup: 0.17x
